In [ ]:
# Este código crea la carpeta labels que se divide en test y train porque YOLO necesita los labels en txt

import os
import xml.etree.ElementTree as ET

DATASET_PATH = "Data/images/images"
OUTPUT_LABELS = "Data/labels"

# Crear carpetas
for split in ["train", "test"]:
    os.makedirs(os.path.join(OUTPUT_LABELS, split), exist_ok=True)

# Extraer todas las clases automáticamente
classes = set()

for split in ["train", "test"]:
    folder = os.path.join(DATASET_PATH, split)
    for file in os.listdir(folder):
        if file.endswith(".xml"):
            tree = ET.parse(os.path.join(folder, file))
            root = tree.getroot()
            for obj in root.iter("object"):
                classes.add(obj.find("name").text)

classes = sorted(list(classes))
print("Clases:", classes)

# Guardar clases
with open("classes.txt", "w") as f:
    for c in classes:
        f.write(c + "\n")


def convert(size, box):
    dw = 1. / size[0]
    dh = 1. / size[1]

    x = (box[0] + box[1]) / 2.0
    y = (box[2] + box[3]) / 2.0
    w = box[1] - box[0]
    h = box[3] - box[2]

    return (x * dw, y * dh, w * dw, h * dh)


for split in ["train", "test"]:
    folder = os.path.join(DATASET_PATH, split)
    out_folder = os.path.join(OUTPUT_LABELS, split)

    for file in os.listdir(folder):
        if not file.endswith(".xml"):
            continue

        tree = ET.parse(os.path.join(folder, file))
        root = tree.getroot()

        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)

        txt_file = file.replace(".xml", ".txt")
        with open(os.path.join(out_folder, txt_file), "w") as f:

            for obj in root.iter("object"):
                cls = obj.find("name").text
                cls_id = classes.index(cls)

                xmlbox = obj.find("bndbox")
                b = (
                    float(xmlbox.find("xmin").text),
                    float(xmlbox.find("xmax").text),
                    float(xmlbox.find("ymin").text),
                    float(xmlbox.find("ymax").text),
                )

                bb = convert((w, h), b)
                f.write(f"{cls_id} {' '.join(map(str, bb))}\n")

Clases: ['Ace of Clubs', 'Ace of Diamonds', 'Ace of Hearts', 'Ace of Spades', 'Jack of Clubs', 'Jack of Diamonds', 'Jack of Hearts', 'Jack of Spades', 'Joker', 'King of Clubs', 'King of Diamonds', 'King of Hearts', 'King of Spades', 'Queen of Clubs', 'Queen of Diamonds', 'Queen of Hearts', 'Queen of Spades']


Por cada imagen:

IMG_123.jpg
IMG_123.txt

El .txt contiene:

<class_id> <x_center> <y_center> <width> <height>